# Notebook 10 — Panel Extension: Instruments, External Validity, and ML
## Extension of Saadaoui (2026, JCE)

This notebook contains everything related to multi-dyad analysis and panel ML.
It supersedes all previous panel notebooks (10_panel_dyads, 10b, 10_fixed, 00_diagnostic).

| Section | Content | Plan step |
|---------|---------|----------|
| A | Instrument diagnostic — all dyads, all candidate instruments | 7.1 |
| B | Dyad-by-dyad LP-IV (valid dyads only) | 7.2 |
| C | Control-function pooled panel | 7.2 |
| D | Panel DML-PLIV | 3.2 + 7.2 |
| E | Multi-outcome panel | 6.2 |
| F | Dyad heterogeneity and hypothesis tests | 7.2 |
| G | PRI co-movement network | 8 |
| H | Power analysis | diagnostic |

### On GDELT controls
NB05 excluded GDELT from estimation citing pre-2000 sparsity. Re-examination shows:
- `gdelt_goldstein_mean` range: [-0.733, 4.789], 386/386 months, bilateral US-China
- `gdelt_sentiment_signal`, `gdelt_total_events_log`: full coverage, not sparse
- These variables were built from bilateral US-China GDELT events (CAMEO-coded)

We therefore add `gdelt_goldstein_mean` and `gdelt_sentiment_signal` to `controls_geopol`
in this notebook. We test whether they enter the DML nuisance functions significantly
(SHAP importance > 0). This is the NLP contribution the professor's plan described.


In [1]:
from pathlib import Path
import warnings, json, time
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tools.tools import add_constant
from statsmodels.tsa.stattools import grangercausalitytests
from linearmodels.iv import IV2SLS
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from scipy import stats
import doubleml as dml
from xgboost import XGBRegressor
import shap

warnings.filterwarnings('ignore')
np.random.seed(42)

cwd  = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
FINAL   = ROOT / 'data' / 'final'
NLP_DIR = ROOT / 'data' / '03_nlp'
RAW     = ROOT / 'data' / 'raw'
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'
for d in [RESULTS, FIGURES]: d.mkdir(parents=True, exist_ok=True)

HMAX = 48
print(f'ROOT={ROOT}')


ROOT=C:\Users\HP\Desktop\replication+contribution


In [2]:
# ── Load base data ─────────────────────────────────────────────────────────────
df_ext = pd.read_csv(FINAL / 'df_extended.csv', index_col=0, parse_dates=True)
df_ext.index = pd.to_datetime(df_ext.index)

with open(FINAL / 'variable_roles.json') as f:
    roles = json.load(f)

INSTRUMENT = roles['instrument_core'][0]   # d2pri
TREATMENT  = roles['treatment'][0]          # lpri
OUTCOME    = roles['outcome'][0]            # lwti
CONTROLS_BASE = (roles['controls_core'] +
                 roles['controls_macro'] +
                 roles['controls_geopol'])
assert 'l2lwip' not in CONTROLS_BASE

# ── Add GDELT NLP controls ─────────────────────────────────────────────────────
# NB03 built bilateral US-China GDELT features with 100% coverage.
# NB05 excluded them citing sparsity, but Goldstein mean and sentiment signal
# have no zeros and range [-0.7, 4.8] / [0.1, 27.9] — not sparse.
# We add two carefully chosen GDELT variables:
#   gdelt_goldstein_mean: expert-coded valence (-10 conflict to +10 cooperation)
#   gdelt_sentiment_signal: |Goldstein| × log(events) — captures certainty of tone
# These are lagged t-1 in feature_matrix_nlp_A.csv (NB03 applied shift(1)).

NLP_CONTROLS = []
nlp_path = NLP_DIR / 'feature_matrix_nlp_A.csv'
if nlp_path.exists():
    df_nlp = pd.read_csv(nlp_path, index_col=0, parse_dates=True)
    df_nlp.index = pd.to_datetime(df_nlp.index)
    gdelt_cols = ['gdelt_goldstein_mean', 'gdelt_sentiment_signal', 'gdelt_total_events_log']
    available  = [c for c in gdelt_cols if c in df_nlp.columns]
    if available:
        df_ext = df_ext.join(df_nlp[available], how='left')
        NLP_CONTROLS = available
        print(f'GDELT NLP controls added: {NLP_CONTROLS}')
        for c in NLP_CONTROLS:
            n = df_ext[c].notna().sum()
            print(f'  {c}: {n}/386 obs | range [{df_ext[c].min():.3f}, {df_ext[c].max():.3f}]')
    else:
        print('GDELT columns not found in feature_matrix_nlp_A.csv')
else:
    print(f'NLP feature matrix not found at {nlp_path}')
    print('Run Notebook 03 first.')

CONTROLS_NLP = CONTROLS_BASE + NLP_CONTROLS  # extended with GDELT
CONTROLS = CONTROLS_NLP  # use NLP-enriched controls throughout

print(f'\nControl set: {len(CONTROLS)} variables')
print(f'  Base: {len(CONTROLS_BASE)} | NLP added: {len(NLP_CONTROLS)}')
print(f'\ndf_extended: n={len(df_ext)} | {df_ext.index.min().date()} to {df_ext.index.max().date()}')


GDELT NLP controls added: ['gdelt_goldstein_mean', 'gdelt_sentiment_signal', 'gdelt_total_events_log']
  gdelt_goldstein_mean: 0/386 obs | range [nan, nan]
  gdelt_sentiment_signal: 0/386 obs | range [nan, nan]
  gdelt_total_events_log: 0/386 obs | range [nan, nan]

Control set: 17 variables
  Base: 14 | NLP added: 3

df_extended: n=385 | 1990-02-28 to 2022-02-28


In [3]:
# ── Load Stata file for all dyad PRI series ────────────────────────────────────
stata_candidates = (list(RAW.glob('*.dta')) + list(ROOT.glob('*.dta')) +
                    list(Path('.').glob('**/*.dta')))
assert stata_candidates, 'Stata .dta not found'
df_raw = pd.read_stata(stata_candidates[0])

date_cols = [c for c in df_raw.columns
             if 'date' in c.lower() or pd.api.types.is_datetime64_any_dtype(df_raw[c])]
if date_cols:
    df_raw['_date'] = pd.to_datetime(df_raw[date_cols[0]])
else:
    for c in df_raw.columns:
        try:
            d = pd.to_datetime('1960-01-01') + pd.to_timedelta(df_raw[c], unit='D')
            if d.dt.year.between(1985, 2025).all(): df_raw['_date'] = d; break
        except: pass
df_raw = df_raw.set_index('_date').sort_index()
df_raw.index = df_raw.index.to_period('M').to_timestamp('M')

# Build d2pri for dyads that only have dlpri
DYAD_DEFS = [
    ('us',    'US-China',       'lpri',       'd2pri',     'dlpri'),
    ('jp',    'Japan-China',    'lpri_jp',    'd2pri_jp',  'dlpri_jp'),
    ('aus',   'Australia-Ch.', 'lpri_aus',   None,        'dlpri_aus'),
    ('cds',   'S.Korea-China', 'lpri_cds',   None,        'dlpri_cds'),
    ('fra',   'France-China',  'lpri_fra',   None,        'dlpri_fra'),
    ('ger',   'Germany-China', 'lpri_ger',   None,        'dlpri_ger'),
    ('india', 'India-China',   'lpri_india', None,        'dlpri_india'),
    ('indo',  'Indonesia-Ch.','lpri_indo',   None,        'dlpri_indo'),
    ('pak',   'Pakistan-Ch.', 'lpri_pak',    None,        'dlpri_pak'),
    ('rus',   'Russia-China',  'lpri_rus',   None,        'dlpri_rus'),
    ('vn',    'Vietnam-China', 'lpri_vn',    None,        'dlpri_vn'),
    ('uk',    'UK-China',      'lpri_uk',    None,        'dlpri_uk'),
]
for code, name, lpri, d2pri, dlpri in DYAD_DEFS:
    if d2pri is None and dlpri in df_raw.columns:
        df_raw[f'd2pri_{code}'] = df_raw[dlpri].diff()

print(f'Stata: {df_raw.shape} | {df_raw.index.min().date()} to {df_raw.index.max().date()}')


Stata: (386, 59) | 1990-01-31 to 2022-02-28


---
## Section A: Instrument Diagnostic — All Dyads

Tests four candidate instruments for each dyad:
1. `d2pri` — Saadaoui's second difference (already in Stata for US, JP)
2. `dlpri` — first difference (contemporaneous)
3. `L1dlpri` — lagged first difference (predetermined)
4. `L2dlpri` — two-period lagged first difference

**Critical bug fix from previous version:** The `first_stage_F` function previously
included lags of the endogenous variable as controls, causing near-perfect
multicollinearity with `dlpri` (since `dlpri = lpri_t - lpri_{t-1}` and L1_lpri
is also `lpri_{t-1}`). Fix: use only macro controls, not endog lags, for the
instrument-relevance test. Endog lags are included in the main estimation but
not in the relevance diagnostic.


In [4]:
def first_stage_F_safe(endog, instr, ctrl_df):
    """
    First-stage F WITHOUT endog lags (avoids multicollinearity with dlpri).
    Returns (F, n_obs) — F=nan if instrument variance degenerate or overflow.
    """
    df = pd.DataFrame({'endog': endog, 'instr': instr})
    for c in ctrl_df.columns:
        df[c] = ctrl_df[c].values if len(ctrl_df) == len(df) else ctrl_df[c]
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    if df['instr'].std() < 1e-8 or df['endog'].std() < 1e-8 or len(df) < 30:
        return np.nan, len(df)
    X = add_constant(df[['instr'] + ctrl_df.columns.tolist()], has_constant='add')
    try:
        fit = sm.OLS(df['endog'], X).fit(cov_type='HC1')
        F   = float(fit.f_test('instr = 0').fvalue)
        return (np.nan, len(df)) if F > 1e8 else (F, len(df))
    except:
        return np.nan, len(df)

# Also test exogeneity: does WTI Granger-cause the instrument?
def granger_p(instr_series, wti_series, maxlag=3):
    df_gc = pd.DataFrame({'instr': instr_series.diff(),
                          'dwti':  wti_series.diff()}).dropna()
    try:
        gc = grangercausalitytests(df_gc[['instr','dwti']], maxlag=maxlag, verbose=False)
        return min(gc[lag][0]['ssr_ftest'][1] for lag in range(1, maxlag+1))
    except:
        return np.nan

ctrl_df_diag = df_ext[roles['controls_core']].copy()

print('INSTRUMENT DIAGNOSTIC — ALL DYADS (bug-fixed: no endog lags in F test)')
print('='*100)
print(f'  {"Dyad":<20} {"F(d2pri)":>10} {"F(dlpri)":>10} {"F(L1dlpri)":>12}',
      f'{"F(L2dlpri)":>12} {"Exog_p":>8}')
print('-'*88)

diag_rows = []
for code, name, lpri_col, d2pri_orig, dlpri_col in DYAD_DEFS:
    if lpri_col not in df_raw.columns or dlpri_col not in df_raw.columns:
        print(f'  {name:<20} MISSING'); continue

    d2pri_col = d2pri_orig if d2pri_orig and d2pri_orig in df_raw.columns else f'd2pri_{code}'
    endog  = df_raw[lpri_col].reindex(ctrl_df_diag.index)
    dlpri  = df_raw[dlpri_col].reindex(ctrl_df_diag.index)
    d2pri  = df_raw[d2pri_col].reindex(ctrl_df_diag.index) if d2pri_col in df_raw.columns else dlpri.diff()

    F_d2,  n1 = first_stage_F_safe(endog, d2pri,         ctrl_df_diag)
    F_dl,  n2 = first_stage_F_safe(endog, dlpri,         ctrl_df_diag)
    F_L1,  n3 = first_stage_F_safe(endog, dlpri.shift(1),ctrl_df_diag)
    F_L2,  n4 = first_stage_F_safe(endog, dlpri.shift(2),ctrl_df_diag)
    exog_p    = granger_p(dlpri, df_ext[OUTCOME])

    def fmt(F):
        if pd.isna(F): return 'OVERFLOW'
        return f'{F:.1f}{"✓" if F>=10 else "⚠" if F>=5 else "✗"}'

    print(f'  {name:<20} {fmt(F_d2):>10} {fmt(F_dl):>10} {fmt(F_L1):>12}',
          f'{fmt(F_L2):>12} {exog_p:>8.3f}')

    # Assign best instrument
    best_F, best_instr, best_col = 0, None, None
    for F_val, instr_name, col in [
        (F_d2, 'd2pri', d2pri_col),
        (F_dl, 'dlpri', dlpri_col),
        (F_L1, 'L1dlpri', dlpri_col+'_L1'),
        (F_L2, 'L2dlpri', dlpri_col+'_L2'),
    ]:
        if not pd.isna(F_val) and F_val > best_F:
            best_F, best_instr, best_col = F_val, instr_name, col

    # Build the actual instrument series for this dyad
    if best_instr == 'd2pri':    instr_series = d2pri
    elif best_instr == 'dlpri':  instr_series = dlpri
    elif best_instr == 'L1dlpri': instr_series = dlpri.shift(1)
    elif best_instr == 'L2dlpri': instr_series = dlpri.shift(2)
    else: instr_series = None

    valid = (best_F >= 10) and (not pd.isna(exog_p)) and (exog_p >= 0.05)
    diag_rows.append({
        'code':code, 'name':name, 'lpri_col':lpri_col,
        'F_d2':F_d2, 'F_dl':F_dl, 'F_L1':F_L1, 'F_L2':F_L2,
        'best_instr':best_instr, 'best_F':best_F,
        'exog_p':exog_p, 'valid':valid,
        'instr_series': instr_series,
    })

diag_df = pd.DataFrame([{k:v for k,v in r.items() if k != 'instr_series'}
                          for r in diag_rows])
diag_df.to_csv(RESULTS / 'instrument_diagnostic.csv', index=False)

print()
valid_dyads = [r for r in diag_rows if r['valid']]
print(f'Valid dyads (F≥10 AND exog_p≥0.05): {len(valid_dyads)}/12')
for r in valid_dyads:
    print(f'  {r["name"]:<22}: best={r["best_instr"]} F={r["best_F"]:.1f}')


INSTRUMENT DIAGNOSTIC — ALL DYADS (bug-fixed: no endog lags in F test)
  Dyad                   F(d2pri)   F(dlpri)   F(L1dlpri)   F(L2dlpri)   Exog_p
----------------------------------------------------------------------------------------
  US-China                   0.1✗       8.1⚠         9.3⚠        11.0✓    0.478
  Japan-China                0.3✗       4.2✗         9.2⚠         9.8⚠    0.242
  Australia-Ch.              0.0✗      53.3✓        63.2✓        71.0✓    0.129
  S.Korea-China              0.1✗       0.0✗         0.0✗         0.0✗    0.783
  France-China               0.0✗       2.2✗         2.2✗         2.2✗    0.563
  Germany-China              0.0✗      13.8✓        16.0✓        13.3✓    0.639
  India-China                0.0✗       0.9✗         1.1✗         1.8✗    0.614
  Indonesia-Ch.              0.1✗       7.2⚠         7.7⚠         9.4⚠    0.073
  Pakistan-Ch.               0.0✗       3.3✗         2.7✗         2.8✗    0.323
  Russia-China               0.0✗      2

---
## Section B: Dyad-by-Dyad LP-IV (Valid Dyads)


In [5]:
def F_shift(s, h): return s.shift(-h)

def add_lags(df, y_col, endog_col, y_lags=3, endog_lags=2):
    out = df.copy(); lag_cols = []
    for l in range(1, y_lags+1):
        c = f'L{l}_{y_col}'; out[c] = out[y_col].shift(l); lag_cols.append(c)
    for l in range(1, endog_lags+1):
        c = f'L{l}_{endog_col}'; out[c] = out[endog_col].shift(l); lag_cols.append(c)
    return out, lag_cols

def lp_iv_single(df_base, endog_series, instr_series, controls, outcome=OUTCOME, hmax=HMAX):
    """
    LP-IV for a single dyad.
    endog_series, instr_series: pd.Series aligned to df_base index.
    """
    work = df_base[[outcome] + controls].copy()
    work['__endog__'] = endog_series
    work['__instr__'] = instr_series
    work, lag_cols = add_lags(work, outcome, '__endog__')
    exog_cols = lag_cols + controls

    rows = []
    for h in range(hmax + 1):
        hdf = pd.DataFrame({
            'y_fwd': F_shift(work[outcome], h),
            'endog': work['__endog__'],
            'instr': work['__instr__'],
            **{c: work[c] for c in exog_cols},
        }).replace([np.inf,-np.inf], np.nan).dropna()
        if len(hdf) < 40:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(hdf),'F':np.nan})
            continue
        try:
            X_fs = add_constant(hdf[['instr']+exog_cols], has_constant='add')
            fs = sm.OLS(hdf['endog'], X_fs).fit(cov_type='HC1')
            F_val = float(fs.f_test('instr = 0').fvalue)
            if F_val > 1e8: F_val = np.nan
            fit = IV2SLS(dependent=hdf['y_fwd'],
                         exog=add_constant(hdf[exog_cols], has_constant='add'),
                         endog=hdf['endog'], instruments=hdf['instr']
                         ).fit(cov_type='robust', debiased=True)
            rows.append({'h':h,'coef':float(fit.params.get('endog',np.nan)),
                         'se':float(fit.std_errors.get('endog',np.nan)),
                         'n':len(hdf),'F':F_val})
        except:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(hdf),'F':np.nan})

    irf = pd.DataFrame(rows)
    irf['lo90'] = irf['coef'] - 1.645*irf['se']
    irf['hi90'] = irf['coef'] + 1.645*irf['se']
    return irf

print('Running dyad-by-dyad LP-IV for valid dyads...')
irf_by_dyad = {}
for spec in valid_dyads:
    print(f'  {spec["name"]}...', end=' ')
    endog = df_raw[spec['lpri_col']].reindex(df_ext.index)
    instr = spec['instr_series'].reindex(df_ext.index)
    irf   = lp_iv_single(df_ext, endog, instr, CONTROLS)
    irf_by_dyad[spec['code']] = irf
    irf.to_csv(RESULTS / f'irf_dyad_{spec["code"]}.csv', index=False)
    sig90 = (irf['lo90']>0).sum() + (irf['hi90']<0).sum()
    print(f'F_mean={irf["F"].mean():.1f}  sig90={sig90}/{HMAX+1}')


Running dyad-by-dyad LP-IV for valid dyads...
  US-China... F_mean=nan  sig90=0/49
  Australia-Ch.... F_mean=nan  sig90=0/49
  Germany-China... F_mean=nan  sig90=0/49
  Russia-China... F_mean=nan  sig90=0/49


In [6]:
# ── Dyad IRF figure ────────────────────────────────────────────────────────────
hs = np.arange(HMAX+1)
n_valid = len(valid_dyads)
ncols = min(3, n_valid)
nrows = int(np.ceil(n_valid/ncols)) if n_valid > 0 else 1
fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 5*nrows))
axes = np.array(axes).flatten() if n_valid > 1 else [axes]

COLORS = ['steelblue','firebrick','darkorange','teal','purple','darkgreen',
          'brown','olive','gray','navy','darkred','black']
us_irf = irf_by_dyad.get('us')

for i, spec in enumerate(valid_dyads):
    ax = axes[i]
    irf = irf_by_dyad[spec['code']]
    col = COLORS[i % len(COLORS)]
    sig90 = (irf['lo90']>0).sum() + (irf['hi90']<0).sum()

    ax.plot(hs, irf['coef'], color=col, lw=2.0, label=spec['name'])
    ax.fill_between(hs, irf['lo90'], irf['hi90'], color=col, alpha=0.18)
    if us_irf is not None and spec['code'] != 'us':
        ax.plot(hs, us_irf['coef'], color='steelblue', lw=0.9,
                linestyle='--', alpha=0.4, label='US-China')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlim(0,HMAX); ax.set_xticks(np.arange(0,HMAX+1,12))
    ax.set_xlabel('Months'); ax.set_ylabel('IRF log WTI')
    ax.set_title(f'{spec["name"]}  inst={spec["best_instr"]}  '
                 f'F={spec["best_F"]:.0f}  sig90={sig90}/{HMAX+1}', fontsize=9)
    ax.legend(fontsize=7)

for j in range(i+1, len(axes)): axes[j].set_visible(False)

plt.suptitle('Section B: Dyad-by-Dyad LP-IV (valid instruments only)\n'
             'Blue dashed = US-China baseline', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES/'Figure_10B_dyad_irfs.png', dpi=300, bbox_inches='tight')
plt.close()
print('Saved: Figure_10B_dyad_irfs.png')


Saved: Figure_10B_dyad_irfs.png


---
## Section C: Control-Function Pooled Panel

When different dyads use different instruments, a pooled IV2SLS with one instrument
is not valid. The control function (CF) approach (Wooldridge 2015) solves this:

1. **First stage per dyad:** regress lpri on the dyad-specific instrument + controls
   → save residuals v̂_d
2. **Pooled second stage:** regress y_fwd on lpri + controls + dyad_FE + v̂_d
   → the residual controls for endogeneity

The coefficient on lpri is the causal average effect.
Standard errors: HC-robust (clustered by dyad infeasible with <20 clusters).


In [7]:
def control_function_panel(df_base, valid_dyads, df_raw_data, controls,
                            outcome=OUTCOME, hmax=HMAX):
    """
    Control function pooled panel LP-IV.
    Uses dyad-specific instruments, pools in second stage with CF correction.
    """
    rows = []
    for h in range(hmax + 1):
        frames = []
        for spec in valid_dyads:
            df_d = df_base[[outcome] + controls].copy()
            endog_s = df_raw_data[spec['lpri_col']].reindex(df_base.index)
            instr_s = spec['instr_series'].reindex(df_base.index)
            df_d['lpri_d']  = endog_s
            df_d['instr_d'] = instr_s
            df_d['dyad']    = spec['code']

            # Lags within dyad
            for l in range(1,4): df_d[f'L{l}_{outcome}'] = df_d[outcome].shift(l)
            for l in range(1,3): df_d[f'L{l}_lpri']      = df_d['lpri_d'].shift(l)
            df_d['y_fwd'] = F_shift(df_d[outcome], h)
            frames.append(df_d)

        panel = pd.concat(frames).replace([np.inf,-np.inf], np.nan)
        lag_cols_p = [f'L{l}_{outcome}' for l in range(1,4)] + \
                     [f'L{l}_lpri' for l in range(1,3)]
        need = ['y_fwd','lpri_d','instr_d','dyad'] + controls + lag_cols_p
        panel = panel[need].dropna()
        if len(panel) < 80:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(panel)}); continue

        try:
            # First stage PER DYAD → residuals
            panel['cf_resid'] = np.nan
            for d in panel['dyad'].unique():
                mask = panel['dyad'] == d
                sd = panel[mask]
                X_fs = add_constant(sd[['instr_d']+controls+lag_cols_p], has_constant='add')
                fs   = sm.OLS(sd['lpri_d'], X_fs).fit()
                panel.loc[mask, 'cf_resid'] = fs.resid

            panel = panel.dropna(subset=['cf_resid'])

            # Dyad dummies (FE)
            dummies = pd.get_dummies(panel['dyad'], prefix='d', drop_first=True).astype(float)
            dum_cols = dummies.columns.tolist()
            panel = pd.concat([panel.reset_index(drop=True),
                               dummies.reset_index(drop=True)], axis=1)

            # Second stage OLS (CF correction)
            X2 = add_constant(
                panel[['lpri_d','cf_resid'] + controls + lag_cols_p + dum_cols],
                has_constant='add')
            fit2 = sm.OLS(panel['y_fwd'], X2).fit(cov_type='HC1')
            rows.append({'h':h,
                         'coef': float(fit2.params.get('lpri_d', np.nan)),
                         'se':   float(fit2.bse.get('lpri_d', np.nan)),
                         'n':    len(panel)})
        except Exception as e:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(panel)})

    irf = pd.DataFrame(rows)
    irf['lo90'] = irf['coef'] - 1.645*irf['se']
    irf['hi90'] = irf['coef'] + 1.645*irf['se']
    return irf

if len(valid_dyads) >= 2:
    print(f'Running control-function panel LP-IV ({len(valid_dyads)} dyads)...')
    irf_cf = control_function_panel(df_ext, valid_dyads, df_raw, CONTROLS)
    irf_cf.to_csv(RESULTS/'irf_panel_cf.csv', index=False)
    sig_cf = (irf_cf['lo90']>0).sum() + (irf_cf['hi90']<0).sum()
    print(f'CF panel: sig90={sig_cf}/{HMAX+1} | n_mean={irf_cf["n"].mean():.0f}')
else:
    print('Not enough valid dyads for panel. Running US-China only.')
    irf_cf = None


Running control-function panel LP-IV (4 dyads)...
CF panel: sig90=0/49 | n_mean=0


---
## Section D: Panel DML-PLIV

DML on the pooled panel with GDELT-enriched controls.
Uses US-China instrument (d2pri) as the primary IV.
For the panel, we restrict to dyads where the instrument is valid.

**Key question:** At n≈(valid dyads × 385), does DML reject the linear IV?
If yes → non-linearity detected at panel scale. If no → linear IV is robust.

**GDELT test:** SHAP values for the DML nuisance functions reveal whether
gdelt_goldstein_mean and gdelt_sentiment_signal carry predictive signal.


In [11]:
def get_xgb(): return XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                                    subsample=0.8, colsample_bytree=0.8,
                                    verbosity=0, random_state=42, n_jobs=-1)

if len(valid_dyads) >= 2:
    # Build panel for DML
    panel_dml_frames = []
    for spec in valid_dyads:
        df_d = df_ext[[OUTCOME] + CONTROLS].copy()
        df_d['lpri_p']  = df_raw[spec['lpri_col']].reindex(df_ext.index)
        df_d['instr_p'] = spec['instr_series'].reindex(df_ext.index)
        df_d['dyad']    = spec['code']
        for l in range(1,4): df_d[f'L{l}_{OUTCOME}'] = df_d[OUTCOME].shift(l)
        for l in range(1,3): df_d[f'L{l}_lpri']      = df_d['lpri_p'].shift(l)
        panel_dml_frames.append(df_d)

    panel_dml = pd.concat(panel_dml_frames)
    lag_p = [f'L{l}_{OUTCOME}' for l in range(1,4)] + [f'L{l}_lpri' for l in range(1,3)]
    dum_dml = pd.get_dummies(panel_dml['dyad'], prefix='d', drop_first=True).astype(float)
    panel_dml = pd.concat([panel_dml.reset_index(drop=True),
                           dum_dml.reset_index(drop=True)], axis=1)
    X_COLS_DML = lag_p + CONTROLS + dum_dml.columns.tolist()

    print(f'Panel DML: {len(panel_dml)} obs | {len(valid_dyads)} dyads | {len(X_COLS_DML)} features')
    print(f'Running DML-PLIV (h=0..{HMAX})...')

    rows_dml = []
    for h in range(HMAX+1):
        if h % 6 == 0: print(f'  h={h}...', end=' ', flush=True)
        panel_dml['y_fwd'] = panel_dml.groupby('dyad')[OUTCOME].shift(-h) \
            if 'dyad' in panel_dml.columns \
            else F_shift(panel_dml[OUTCOME], h)
        sub = panel_dml[['y_fwd','lpri_p','instr_p']+X_COLS_DML].replace(
            [np.inf,-np.inf], np.nan).dropna()
        if len(sub) < 100:
            rows_dml.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(sub)}); continue
        try:
            data_obj = dml.DoubleMLData(sub, y_col='y_fwd', d_cols='lpri_p',
                                         z_cols='instr_p', x_cols=X_COLS_DML)
            pliv = dml.DoubleMLPLIV(data_obj, ml_l=get_xgb(), ml_m=get_xgb(),
                                     ml_r=get_xgb(), n_folds=5, n_rep=3)
            pliv.fit()
            rows_dml.append({'h':h,'coef':float(pliv.coef[0]),'se':float(pliv.se[0]),'n':len(sub)})
        except:
            rows_dml.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(sub)})
    print('done.')

    irf_dml_panel = pd.DataFrame(rows_dml)
    irf_dml_panel['lo90'] = irf_dml_panel['coef'] - 1.645*irf_dml_panel['se']
    irf_dml_panel['hi90'] = irf_dml_panel['coef'] + 1.645*irf_dml_panel['se']
    irf_dml_panel.to_csv(RESULTS/'irf_panel_dml.csv', index=False)

    sig_dml = (irf_dml_panel['lo90']>0).sum() + (irf_dml_panel['hi90']<0).sum()
    print(f'Panel DML: sig90={sig_dml}/{HMAX+1}')

    # SHAP for GDELT controls in the DML nuisance
    if NLP_CONTROLS:
        print('\nSHAP analysis for GDELT controls in DML first-stage nuisance...')
        sub_shap = panel_dml[['lpri_p'] + X_COLS_DML].replace([np.inf, -np.inf], np.nan).dropna()
        X_shap = sub_shap[X_COLS_DML]
        y_shap = sub_shap['lpri_p']
        
        m_shap = get_xgb()
        m_shap.fit(X_shap, y_shap)
        
        # Use the modern SHAP API (works with shap>=0.40)
        try:
            # shap.Explainer is the recommended interface
            explainer = shap.Explainer(m_shap, X_shap, algorithm='tree')
            shap_values = explainer(X_shap).values
        except (AttributeError, TypeError):
            # Fallback for older shap versions (<0.40)
            explainer = shap.TreeExplainer(m_shap)
            shap_values = explainer.shap_values(X_shap)
            # Handle legacy output shape
            if isinstance(shap_values, list):
                shap_values = shap_values[0]
            elif shap_values.ndim == 3:
                shap_values = shap_values[:, 0, :]
        
        # shap_values should now be 2D: (samples, features)
        mean_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=X_shap.columns)
        
        print('SHAP importance (GDELT vs macro controls):')
        for c in NLP_CONTROLS:
            if c in mean_shap.index:
                rank = int((mean_shap >= mean_shap[c]).sum())
                print(f'  {c}: SHAP={mean_shap[c]:.4f} | rank={rank}/{len(mean_shap)}')
        mean_shap.to_csv(RESULTS / 'shap_panel_dml.csv')
        
        if any(mean_shap.get(c, 0) > 0.001 for c in NLP_CONTROLS):
            print('FINDING: GDELT controls carry signal in DML nuisance functions.')
            print('  NLP measurement contributes to non-parametric control.')
        else:
            print('FINDING: GDELT SHAP near zero — no additional signal over macro controls.')

Panel DML: 1540 obs | 4 dyads | 25 features
Running DML-PLIV (h=0..48)...
  h=0...   h=6...   h=12...   h=18...   h=24...   h=30...   h=36...   h=42...   h=48... done.
Panel DML: sig90=0/49

SHAP analysis for GDELT controls in DML first-stage nuisance...
SHAP importance (GDELT vs macro controls):
  gdelt_goldstein_mean: SHAP=nan | rank=0/25
  gdelt_sentiment_signal: SHAP=nan | rank=0/25
  gdelt_total_events_log: SHAP=nan | rank=0/25
FINDING: GDELT SHAP near zero — no additional signal over macro controls.


---
## Section E: Multi-Outcome Panel

Same control-function panel, different outcomes: Brent, gold, VIX, CNY/USD.
Tests whether the average PRI effect generalises beyond WTI.


In [13]:
OUTCOMES_PANEL = [
    ('lwti',    'WTI crude'),
    ('brent',   'Brent crude'),
    ('gold',    'Gold'),
    ('vix',     'VIX'),
    ('cny_usd', 'CNY/USD'),
]
# Only outcomes present in df_ext
OUTCOMES_PANEL = [(c,l) for c,l in OUTCOMES_PANEL if c in df_ext.columns]

if len(valid_dyads) >= 2:
    print('SECTION E: Multi-Outcome Panel LP-IV (control function)')
    print(f'Outcomes: {[l for _,l in OUTCOMES_PANEL]}')
    print()

    irf_multi = {}
    for out_col, out_label in OUTCOMES_PANEL:
        print(f'  {out_label}...', end=' ')
        # ... (data preparation for use_col) ...
        irf_o = control_function_panel(df_ext, valid_dyads, df_raw, CONTROLS,
                                        outcome=use_col)
        irf_multi[out_col] = irf_o
        sig_o = (irf_o['lo90']>0).sum() + (irf_o['hi90']<0).sum()
        
        # --- FIX: handle empty 'coef' column ---
        coef_non_na = irf_o.dropna(subset=['coef'])['coef'].abs()
        if not coef_non_na.empty:
            peak_h = int(coef_non_na.idxmax())
        else:
            peak_h = -1  # or np.nan
            print(f'Warning: no valid coefficients for {out_label}')
    # ----------------------------------------
    
    print(f'sig90={sig_o}/{HMAX+1}  peak_h={peak_h}')
    irf_o.to_csv(RESULTS/f'irf_panel_cf_{out_col}.csv', index=False)

    # Figure
    fig, axes = plt.subplots(1, len(OUTCOMES_PANEL), figsize=(6*len(OUTCOMES_PANEL), 5))
    if len(OUTCOMES_PANEL) == 1: axes = [axes]
    hs = np.arange(HMAX+1)
    for ax, (out_col, out_label) in zip(axes, OUTCOMES_PANEL):
        irf_o = irf_multi[out_col]
        ax.plot(hs, irf_o['coef'], lw=2.0, label=out_label)
        ax.fill_between(hs, irf_o['lo90'], irf_o['hi90'], alpha=0.18)
        ax.axhline(0, color='black', lw=0.8)
        ax.set_title(out_label, fontsize=10)
        ax.set_xlabel('Months'); ax.set_xlim(0,HMAX)
        ax.set_xticks(np.arange(0,HMAX+1,12))
    plt.suptitle('Section E: Multi-Outcome Panel LP-IV\nAverage effect of bilateral PRI across valid dyads',
                 y=1.01, fontsize=11)
    plt.tight_layout()
    plt.savefig(FIGURES/'Figure_10E_multi_outcome_panel.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved: Figure_10E_multi_outcome_panel.png')
else:
    print('Skipped: insufficient valid dyads.')


SECTION E: Multi-Outcome Panel LP-IV (control function)
Outcomes: ['WTI crude', 'Brent crude', 'Gold', 'VIX', 'CNY/USD']

  WTI crude... Warning: no valid coefficients for WTI crude
  Brent crude... Warning: no valid coefficients for Brent crude
  Gold... Warning: no valid coefficients for Gold
  VIX... Warning: no valid coefficients for VIX
  CNY/USD... Warning: no valid coefficients for CNY/USD
sig90=0/49  peak_h=-1
Saved: Figure_10E_multi_outcome_panel.png


---
## Section F: Dyad Heterogeneity — Hypotheses and Tests

**H1 (US-China effect):** The PRI→WTI effect is stronger for US-China than for
the average of other dyads (US-China is a strategically more important dyad).

**H2 (External validity):** Japan-China replicates the US-China IRF pattern
(positive medium-run effect). Wald test: β_US ≈ β_JP.

**H3 (Network structure):** Dyads with lower PRI co-movement with other dyads
(more idiosyncratic) show larger oil price effects (their shocks are not
diversified across other bilateral relationships).


In [14]:
print('SECTION F: Dyad Heterogeneity Hypothesis Tests')
print('='*60)

# H2: Wald test US vs JP
if 'us' in irf_by_dyad and 'jp' in irf_by_dyad:
    wald_rows = []
    for h in range(HMAX+1):
        cu,su = irf_by_dyad['us'].loc[h,'coef'], irf_by_dyad['us'].loc[h,'se']
        cj,sj = irf_by_dyad['jp'].loc[h,'coef'], irf_by_dyad['jp'].loc[h,'se']
        if any(pd.isna([cu,su,cj,sj])) or su==0 or sj==0:
            wald_rows.append({'h':h,'diff':np.nan,'p':np.nan}); continue
        diff = cu-cj; se = np.sqrt(su**2+sj**2)
        W = (diff/se)**2; p = 1-stats.chi2.cdf(W,df=1)
        wald_rows.append({'h':h,'diff':diff,'p':p})
    wald_usjp = pd.DataFrame(wald_rows)
    sig_usjp  = (wald_usjp['p']<0.10).sum()
    print(f'H2 (US vs JP equality): {sig_usjp}/{HMAX+1} significant at 10%')
    if sig_usjp <= int(0.1*(HMAX+1)):
        print('  RESULT: Cannot reject H0 (effects similar).')
        print('  → Japan-China validates US-China. External validity confirmed.')
    else:
        print(f'  RESULT: Effects differ at h={wald_usjp[wald_usjp["p"]<0.10]["h"].tolist()}')
        print('  → Dyad heterogeneity confirmed between US-China and Japan-China.')
    wald_usjp.to_csv(RESULTS/'wald_us_vs_jp.csv', index=False)
    print()

# H3: Peak coefficient vs PRI idiosyncrasy
# Build PRI correlation matrix
pri_df = pd.DataFrame()
for spec in diag_rows:
    if spec['lpri_col'] in df_raw.columns:
        pri_df[spec['name']] = df_raw[spec['lpri_col']].reindex(df_ext.index).diff()
pri_df = pri_df.dropna(how='all')
if len(pri_df.columns) >= 2:
    corr_m = pri_df.corr()
    # Mean absolute correlation with other dyads (lower = more idiosyncratic)
    idiosyncrasy = {}
    for c in corr_m.columns:
        others = corr_m[c].drop(c)
        idiosyncrasy[c] = float(others.abs().mean())

    print('H3: Idiosyncrasy vs peak IRF coefficient')
    print(f'  {"Dyad":<22} {"Mean |r|":>10}  {"Idiosyncratic":>15}  {"Peak coef":>10}')
    h3_rows = []
    for spec in valid_dyads:
        name = spec['name']
        idio = idiosyncrasy.get(name, np.nan)
        irf  = irf_by_dyad.get(spec['code'])
        if irf is not None:
            valid_irf = irf.dropna(subset=['coef'])
            if not valid_irf.empty:
                pk = float(valid_irf['coef'].abs().max())
                print(f'  {name:<22} {idio:>10.3f}  {"high" if idio<0.10 else "low":>15}  {pk:>10.4f}')
                h3_rows.append({'name':name,'idiosyncrasy':idio,'peak_coef':pk})

    if len(h3_rows) >= 3:
        h3_df = pd.DataFrame(h3_rows)
        r, p_h3 = stats.spearmanr(h3_df['idiosyncrasy'], h3_df['peak_coef'])
        print(f'\n  Spearman correlation (idiosyncrasy vs |peak coef|): r={r:.3f} p={p_h3:.3f}')
        if p_h3 < 0.10:
            if r < 0:
                print('  RESULT: More idiosyncratic dyads have larger oil price effects. H3 supported.')
            else:
                print('  RESULT: More correlated dyads have larger effects. Opposite of H3.')
        else:
            print('  RESULT: No significant relationship between idiosyncrasy and effect size.')


SECTION F: Dyad Heterogeneity Hypothesis Tests
H3: Idiosyncrasy vs peak IRF coefficient
  Dyad                     Mean |r|    Idiosyncratic   Peak coef


---
## Section G: PRI Co-movement Network


In [15]:
print('SECTION G: PRI Co-movement Network')
corr_m.to_csv(RESULTS/'pri_correlation_matrix.csv')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
ax = axes[0]
im = ax.imshow(corr_m.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_m.columns))); ax.set_xticklabels(corr_m.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(corr_m.columns))); ax.set_yticklabels(corr_m.columns, fontsize=8)
for i in range(len(corr_m.columns)):
    for j in range(len(corr_m.columns)):
        v = corr_m.iloc[i,j]
        if not pd.isna(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6,
                    color='white' if abs(v)>0.5 else 'black')
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_title('PRI Co-movement (first differences)\nGeopolitical Interdependence Network')

try:
    import networkx as nx
    G = nx.Graph()
    for c in corr_m.columns: G.add_node(c)
    for i,c1 in enumerate(corr_m.columns):
        for j,c2 in enumerate(corr_m.columns):
            if i<j:
                r = corr_m.loc[c1,c2]
                if not pd.isna(r) and abs(r)>0.15:
                    n1_obs = pri_df[c1].notna().sum() if c1 in pri_df else 0
                    n2_obs = pri_df[c2].notna().sum() if c2 in pri_df else 0
                    t = r*np.sqrt(min(n1_obs,n2_obs)-2)/np.sqrt(1-r**2)
                    p = 2*(1-stats.t.cdf(abs(t), df=min(n1_obs,n2_obs)-2))
                    if p < 0.05: G.add_edge(c1,c2,weight=abs(r),positive=(r>0))
    pos = nx.spring_layout(G, seed=42, k=2.5)
    ec  = ['firebrick' if G[u][v]['positive'] else 'steelblue' for u,v in G.edges()]
    ew  = [G[u][v]['weight']*6 for u,v in G.edges()]
    ns  = [200+150*G.degree(n) for n in G.nodes()]
    ax2 = axes[1]
    nx.draw_networkx_nodes(G, pos, ax=ax2, node_color='lightgray', node_size=ns, alpha=0.9)
    nx.draw_networkx_labels(G, pos, ax=ax2, font_size=7)
    nx.draw_networkx_edges(G, pos, ax=ax2, edge_color=ec, width=ew, alpha=0.7)
    ax2.set_title('PRI Co-movement Network\nEdges: |r|>0.15, p<0.05 | Red=positive, Blue=negative')
    ax2.axis('off')
    ax2.text(0.02,0.02,'Red: co-move together\nBlue: opposite directions\nNode size: connections',
             transform=ax2.transAxes, fontsize=8,
             bbox=dict(boxstyle='round',facecolor='white',alpha=0.8))
except ImportError:
    axes[1].text(0.5,0.5,'pip install networkx\nfor network graph',
                 ha='center',va='center', transform=axes[1].transAxes)

plt.suptitle('Section G: Geopolitical Interdependence Network\nPRI co-movement across X-China dyads (ΔPRI)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES/'Figure_10G_network.png', dpi=300, bbox_inches='tight')
plt.close()
print('Saved: Figure_10G_network.png')


SECTION G: PRI Co-movement Network
Saved: Figure_10G_network.png


---
## Section H: Power Analysis


In [16]:
print('SECTION H: Power Analysis')
print('='*60)

alpha_c = stats.norm.ppf(0.90)

# Get empirical SE from US-China regime split
irf_us  = irf_by_dyad.get('us')
if irf_us is not None:
    median_se = irf_us['se'].median()
    SE_diff = np.sqrt(2) * median_se  # approximate for two equal subsamples
else:
    median_se = 0.15
    SE_diff   = np.sqrt(2) * 0.15

print(f'Empirical SE (US-China): {median_se:.4f}')
print(f'SE_diff for regime test: {SE_diff:.4f}')
print()

print(f'{"Test":35s} {"MDE@80%":>10}  Notes')
print('-'*65)

for test, n_per, SE in [
    ('Regime Wald (NB07, n=185/regime)',   185,  SE_diff),
    ('Panel CF (all valid dyads)',          len(valid_dyads)*385//2 if valid_dyads else 385,
     SE_diff * np.sqrt(185/max(1,len(valid_dyads)*385//2))),
]:
    mde = (alpha_c + stats.norm.ppf(0.80)) * SE
    print(f'{test:35s} {mde:>10.4f}  n={n_per}')

# Power curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
delta_g = np.linspace(0, 0.5, 200)
for n_per, col, lab in [
    (185,  'firebrick', 'Regime Wald (n=185)'),
    (385,  'darkorange', 'Full sample (n=385)'),
    (max(1,len(valid_dyads))*185, 'steelblue', f'Panel ({len(valid_dyads)} dyads)'),
    (3000, 'darkgreen', 'Hypothetical n=3000'),
]:
    SE = SE_diff * np.sqrt(185/n_per)
    pw = [1-stats.norm.cdf(alpha_c - d/SE) for d in delta_g]
    axes[0].plot(delta_g, pw, color=col, lw=2, label=lab)
axes[0].axhline(0.80, color='black', lw=1, linestyle='--')
axes[0].set_xlabel('True effect size Δ'); axes[0].set_ylabel('Power')
axes[0].set_title('Power curves'); axes[0].legend(fontsize=8)
axes[0].set_xlim(0,0.5); axes[0].set_ylim(0,1)

n_g = np.arange(50, 3000, 50)
mde_g = [(alpha_c+stats.norm.ppf(0.80))*SE_diff*np.sqrt(185/n) for n in n_g]
axes[1].plot(n_g, mde_g, color='steelblue', lw=2)
axes[1].axhline(0.10, color='firebrick', lw=1.5, linestyle='--', label='Δ=0.10')
axes[1].axvline(185, color='gray', lw=1, linestyle=':')
axes[1].set_xlabel('Sample size per regime'); axes[1].set_ylabel('MDE at 80% power')
axes[1].set_title('MDE vs sample size'); axes[1].legend(fontsize=8)
axes[1].set_xlim(0,2000); axes[1].set_ylim(0,0.6)

plt.suptitle('Section H: Power Analysis\nWhy null results are uninformative, not evidence of no effect',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES/'Figure_10H_power.png', dpi=300, bbox_inches='tight')
plt.close()
print('Saved: Figure_10H_power.png')


SECTION H: Power Analysis
Empirical SE (US-China): nan
SE_diff for regime test: nan

Test                                   MDE@80%  Notes
-----------------------------------------------------------------
Regime Wald (NB07, n=185/regime)           nan  n=185
Panel CF (all valid dyads)                 nan  n=770
Saved: Figure_10H_power.png


In [17]:
print('NOTEBOOK 10 — COMPLETE SUMMARY')
print('='*65)
print()
print('GDELT CONTROLS:')
print(f'  Added: {NLP_CONTROLS}')
if NLP_CONTROLS and irf_dml_panel is not None:
    shap_df = pd.read_csv(RESULTS/'shap_panel_dml.csv', index_col=0)
    for c in NLP_CONTROLS:
        if c in shap_df.index:
            print(f'  {c}: SHAP={shap_df.loc[c,"0"]:.4f}')
print()
print('INSTRUMENT DIAGNOSTIC:')
for r in diag_rows:
    print(f'  {r["name"]:<22}: best={r["best_instr"]} F={r["best_F"]:.1f} valid={r["valid"]}')
print()
print('VALID DYAD LP-IV:')
for spec in valid_dyads:
    irf = irf_by_dyad.get(spec['code'])
    if irf is not None:
        sig90 = (irf['lo90']>0).sum()+(irf['hi90']<0).sum()
        print(f'  {spec["name"]:<22}: sig90={sig90}/{HMAX+1}')
print()
if irf_cf is not None:
    sig_cf = (irf_cf['lo90']>0).sum()+(irf_cf['hi90']<0).sum()
    print(f'CF PANEL: sig90={sig_cf}/{HMAX+1}')
if irf_dml_panel is not None:
    sig_dml = (irf_dml_panel['lo90']>0).sum()+(irf_dml_panel['hi90']<0).sum()
    print(f'DML PANEL: sig90={sig_dml}/{HMAX+1}')


NOTEBOOK 10 — COMPLETE SUMMARY

GDELT CONTROLS:
  Added: ['gdelt_goldstein_mean', 'gdelt_sentiment_signal', 'gdelt_total_events_log']
  gdelt_goldstein_mean: SHAP=nan
  gdelt_sentiment_signal: SHAP=nan
  gdelt_total_events_log: SHAP=nan

INSTRUMENT DIAGNOSTIC:
  US-China              : best=L2dlpri F=11.0 valid=True
  Japan-China           : best=L2dlpri F=9.8 valid=False
  Australia-Ch.         : best=L2dlpri F=71.0 valid=True
  S.Korea-China         : best=d2pri F=0.1 valid=False
  France-China          : best=L2dlpri F=2.2 valid=False
  Germany-China         : best=L1dlpri F=16.0 valid=True
  India-China           : best=L2dlpri F=1.8 valid=False
  Indonesia-Ch.         : best=L2dlpri F=9.4 valid=False
  Pakistan-Ch.          : best=dlpri F=3.3 valid=False
  Russia-China          : best=L2dlpri F=35.1 valid=True
  Vietnam-China         : best=L2dlpri F=1.5 valid=False
  UK-China              : best=L2dlpri F=54.4 valid=False

VALID DYAD LP-IV:
  US-China              : sig90=0/49
  

In [18]:
df_raw.columns.tolist()

['lwip',
 'lgop',
 'lwti',
 'lpri',
 'pri',
 'date',
 'u_wip',
 'u_gop',
 'pri_jp',
 'pri_aus',
 'pri_cds',
 'pri_fra',
 'pri_ger',
 'pri_india',
 'pri_indo',
 'pri_pak',
 'pri_rus',
 'pri_vn',
 'pri_uk',
 'Period',
 'lpri_jp',
 'dlpri_jp',
 'lpri_aus',
 'dlpri_aus',
 'lpri_cds',
 'dlpri_cds',
 'lpri_fra',
 'dlpri_fra',
 'lpri_ger',
 'dlpri_ger',
 'lpri_india',
 'dlpri_india',
 'lpri_indo',
 'dlpri_indo',
 'lpri_pak',
 'dlpri_pak',
 'lpri_rus',
 'dlpri_rus',
 'lpri_vn',
 'dlpri_vn',
 'lpri_uk',
 'dlpri_uk',
 'dlpri',
 'llwip',
 'llgop',
 'l2lwip',
 'l2lgop',
 'd2pri',
 'd2pri_jp',
 'd2pri_aus',
 'd2pri_cds',
 'd2pri_fra',
 'd2pri_ger',
 'd2pri_india',
 'd2pri_indo',
 'd2pri_pak',
 'd2pri_rus',
 'd2pri_vn',
 'd2pri_uk']